# SFT on GSM8K — Chain of Thought

Train Llama-3.2-1B on GSM8K math problems using full chain-of-thought reasoning.

The model learns to reason step by step before producing the final answer.
Compare with `sft_gsm8k_answer_only.ipynb` to see what CoT training actually adds.

In [ ]:
import re
import tinker
from tinker import types
from tinker.types import AdamParams
from datasets import load_dataset
from tinker_cookbook import renderers
from dotenv import load_dotenv
import numpy as np

load_dotenv()

In [ ]:
# Load GSM8K
dataset = load_dataset("openai/gsm8k", "main", split="train")
test_dataset = load_dataset("openai/gsm8k", "main", split="test")

print(f"Train size: {len(dataset)}")
print(f"Test size: {len(test_dataset)}")
print("\nSample example:")
print("Q:", dataset[0]["question"])
print("A:", dataset[0]["answer"])

In [ ]:
def clean_answer(answer: str) -> str:
    """Strip <<calculation>> annotations from GSM8K answers.
    
    Input:  'She sold 48/2 = <<48/2=24>>24 clips.\n#### 24'
    Output: 'She sold 48/2 = 24 clips.\n#### 24'
    """
    return re.sub(r'<<[^>]+>>', '', answer).strip()

def extract_answer(text: str) -> str | None:
    """Extract the final numeric answer after ####."""
    match = re.search(r'####\s*(-?[\d,]+)', text)
    return match.group(1).replace(',', '') if match else None

# Verify
sample = dataset[0]
print("Raw answer:")
print(sample["answer"])
print("\nCleaned answer:")
print(clean_answer(sample["answer"]))
print("\nExtracted final answer:", extract_answer(sample["answer"]))

In [ ]:
service_client = tinker.ServiceClient()

training_client = service_client.create_lora_training_client(
    base_model="meta-llama/Llama-3.2-1B",
    rank=32
)

baseline_sampler = service_client.create_sampling_client(
    base_model="meta-llama/Llama-3.2-1B"
)

tokenizer = training_client.get_tokenizer()
renderer = renderers.get_renderer("llama3", tokenizer)

In [ ]:
def build_messages(example: dict) -> list:
    return [
        {
            "role": "system",
            "content": "You are a math tutor. Solve problems step by step, then give the final answer after ####."
        },
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            # Full chain-of-thought with <<>> annotations stripped
            "content": clean_answer(example["answer"])
        }
    ]

def process_example(example: dict) -> types.Datum:
    messages = build_messages(example)
    model_input, weights = renderer.build_supervised_example(messages)
    tokens = model_input.tolist()
    weights = weights.tolist()
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens[:-1]),
        loss_fn_inputs={
            "target_tokens": tokens[1:],
            "weights": weights[1:]
        }
    )

# Preview one training example
print("Training example assistant turn:")
print(build_messages(dataset[0])[2]["content"])

In [ ]:
N_TRAIN = 500

training_data = [
    process_example(ex)
    for ex in dataset.select(range(N_TRAIN))
]

print(f"Built {len(training_data)} training examples")

In [ ]:
N_STEPS = 50

for step in range(N_STEPS):
    fwd = training_client.forward_backward(
        training_data,
        loss_fn="cross_entropy"
    )
    opt = training_client.optim_step(
        AdamParams(learning_rate=1e-4)
    )

    fwd_res = fwd.result()
    opt.result()

    logprobs = np.concatenate(
        [o["logprobs"].tolist() for o in fwd_res.loss_fn_outputs]
    )
    weights = np.concatenate(
        [d.loss_fn_inputs["weights"].tolist() for d in training_data]
    )

    loss = -np.dot(logprobs, weights) / weights.sum()
    print(f"Step {step:02d} | Loss {loss:.4f}")

In [ ]:
sampler = training_client.save_weights_and_get_sampling_client("gsm8k-sft-cot")

In [ ]:
# Evaluate on held-out test examples
N_EVAL = 100
eval_examples = test_dataset.select(range(N_EVAL))

stop_sequences = renderer.get_stop_sequences()
sampling_params = types.SamplingParams(
    max_tokens=512,
    temperature=0.0,  # greedy for eval
    stop=stop_sequences
)

baseline_correct = 0
trained_correct = 0

for ex in eval_examples:
    prompt = renderer.build_generation_prompt([
        {
            "role": "system",
            "content": "You are a math tutor. Solve problems step by step, then give the final answer after ####."
        },
        {
            "role": "user",
            "content": ex["question"]
        }
    ])

    ground_truth = extract_answer(ex["answer"])

    # Baseline
    baseline_res = baseline_sampler.sample(
        prompt=prompt, num_samples=1, sampling_params=sampling_params
    ).result()
    baseline_response, _ = renderer.parse_response(baseline_res.sequences[0].tokens)
    baseline_pred = extract_answer(baseline_response["content"])
    if baseline_pred == ground_truth:
        baseline_correct += 1

    # Trained
    trained_res = sampler.sample(
        prompt=prompt, num_samples=1, sampling_params=sampling_params
    ).result()
    trained_response, _ = renderer.parse_response(trained_res.sequences[0].tokens)
    trained_pred = extract_answer(trained_response["content"])
    if trained_pred == ground_truth:
        trained_correct += 1

print("=" * 50)
print(f"Eval on {N_EVAL} held-out GSM8K problems")
print("=" * 50)
print(f"Baseline accuracy : {baseline_correct / N_EVAL * 100:.1f}%")
print(f"SFT (CoT) accuracy: {trained_correct / N_EVAL * 100:.1f}%")
print(f"Delta             : +{(trained_correct - baseline_correct) / N_EVAL * 100:.1f}pp")
print("=" * 50)

In [ ]:
# Qualitative comparison on a single example
ex = test_dataset[0]

prompt = renderer.build_generation_prompt([
    {
        "role": "system",
        "content": "You are a math tutor. Solve problems step by step, then give the final answer after ####."
    },
    {
        "role": "user",
        "content": ex["question"]
    }
])

print("QUESTION:", ex["question"])
print("=" * 50)

baseline_res = baseline_sampler.sample(
    prompt=prompt, num_samples=1,
    sampling_params=types.SamplingParams(max_tokens=512, temperature=0.0, stop=stop_sequences)
).result()
baseline_response, _ = renderer.parse_response(baseline_res.sequences[0].tokens)
print("BASELINE:")
print(baseline_response["content"])

print("=" * 50)

trained_res = sampler.sample(
    prompt=prompt, num_samples=1,
    sampling_params=types.SamplingParams(max_tokens=512, temperature=0.0, stop=stop_sequences)
).result()
trained_response, _ = renderer.parse_response(trained_res.sequences[0].tokens)
print("SFT (CoT):")
print(trained_response["content"])

print("=" * 50)
print("GROUND TRUTH:")
print(clean_answer(ex["answer"]))